# Sanity Checks — Wasserstein Distance on Graphs

This notebook verifies the fundamental properties of the $W_2$ graph distance based on EDRep embeddings. Graphs are generated with the **Degree-Corrected Stochastic Block Model (DCSBM)**.

**Properties tested:**

| # | Property | What is checked |
|---|---|---|
| 1 | **Identity** | $d(G, G) = 0$ |
| 2 | **Symmetry** | $d(G_1, G_2) = d(G_2, G_1)$ |
| 3 | **Triangle inequality** | $d(G_1, G_2) + d(G_2, G_3) \geq d(G_1, G_3)$ |
| 4 | **Intra-class permutation invariance** | Shuffling nodes within the same class leaves the distance unchanged |
| 5 | **Sensitivity to partition** | Assigning nodes to the wrong class increases the distance |
| 6 | **Matched formula** | $d = \lVert X X^\top - Y Y^\top \rVert_F / \sqrt{2}$ when each node is its own partition |
| 7 | **Node migration** | Distance grows monotonically as nodes are moved between partitions |

In [6]:
import numpy as np
import networkx as nx
import pandas as pd
import ot
import itertools
import math
import matplotlib.pyplot as plt
import seaborn as sns
from EDRep_main.EDRep import NodeEmbedding
from functions import *
from graph_generators import *


## Imports

---
## 1. Distance Functions

Two core functions used throughout all checks:

| Function | Description |
|---|---|
| `block_dists` | Precomputes sorted 1D value arrays for every bipartite block of the embedding. Intra-class blocks (i = j) use the upper triangle only; inter-class blocks use all products. |
| `graph_distance` | Computes the $\ell^2$ norm of per-block Wasserstein-2 distances between two block-distribution dictionaries. |

In [ ]:
def block_dists(X_list):
    m = len(X_list)
    blocks = {}
    for i in range(m):
        for j in range(i, m):
            A = X_list[i] @ X_list[j].T
            if i == j:
                # intra-class: upper triangle only (symmetric, diagonal excluded)
                if A.shape[0] > 1:
                    vals = A[np.triu_indices_from(A, k=1)]
                else:
                    continue  # single-node class: no pairs to compare
            else:
                # inter-class: all pairwise products
                vals = A.flatten()
            blocks[(i, j)] = np.sort(vals)
    return blocks

def graph_distance(bd_a, bd_b):
    D = []
    for blk, va in bd_a.items():
        vb = bd_b.get(blk)
        if vb is not None:
            D.append(ot.wasserstein_1d(va, vb, p=2) ** 0.5)
    return np.linalg.norm(D)

---
## 2. SBM Graph Generator

`DCSBM` generates graphs whose community structure is controlled by parameter $\alpha$:

$$c_\text{out} = c - \alpha\sqrt{c}, \qquad c_\text{in} = kc - (k-1)\,c_\text{out}$$

- $\alpha = 0$: Erdős–Rényi graph (no community structure)
- Large $\alpha$: sharp communities ($c_\text{in} \gg c_\text{out}$)

`dcsbm_to_nx` converts the edge-list DataFrame to a NetworkX graph with nodes labelled `0…n−1`.

## 3. Graph Generation

Three graphs are generated with $n=1200$ nodes, $k=3$ balanced blocks (400 nodes each), and average degree $c=10$:

| Graph | Model | $\alpha$ | Structure |
|-------|-------|----------|-----------|
| $G_1$ | DCSBM | 1.5      | strong communities |
| $G_2$ | DCSBM | 1.5      | strong communities |
| $G_3$ | DCSBM | 0.5      | weak communities |

$G_1$ and $G_2$ are independent draws from the **same** model → expected small distance.  
$G_3$ has a different structure → expected large distance.

In [ ]:
# DCSBM parameters
n_nodes = 1200
k_sbm = 3
class_sizes = [n_nodes // k_sbm] * k_sbm   # [400, 400, 400]
c_avg = 10
symmetric = True
make_connected = True

theta = np.ones(n_nodes)
l_nodes = np.repeat(np.arange(k_sbm), n_nodes // k_sbm)

# Strong community structure (alpha=1.5)
c_in_1, c_out_1 = compute_cin_cout(c_avg, k_sbm, alpha=1.5)
C_1 = np.full((k_sbm, k_sbm), c_out_1)
np.fill_diagonal(C_1, c_in_1)

# Weaker community structure (alpha=0.5)
c_in_2, c_out_2 = compute_cin_cout(c_avg, k_sbm, alpha=0.5)
C_2 = np.full((k_sbm, k_sbm), c_out_2)
np.fill_diagonal(C_2, c_in_2)

np.random.seed(42); G_1 = dcsbm_to_nx(DCSBM((C_1, c_avg, l_nodes, theta, symmetric, make_connected)))
np.random.seed(43); G_2 = dcsbm_to_nx(DCSBM((C_1, c_avg, l_nodes, theta, symmetric, make_connected)))
np.random.seed(44); G_3 = dcsbm_to_nx(DCSBM((C_2, c_avg, l_nodes, theta, symmetric, make_connected)))

### EDRep Embeddings

Compute `NodeEmbedding` for each graph (`dim=32`, `k=1`) and slice the embedding matrix $X$ according to the ground-truth block partition.

In [ ]:
embedding_dim= 32

def partition(class_sizes):
    part = []
    start = 0
    for i in class_sizes:
        part.append(list(range(start, start + i)))
        start += i
    return part

partitions= partition(class_sizes)
emb_1 = node_embedding(G_1, partitions, embedding_dim)
emb_2 = node_embedding(G_2, partitions, embedding_dim)
emb_3 = node_embedding(G_3, partitions, embedding_dim)


### Precompute distances

Block distributions and pairwise distances are computed once here so that the individual checks in Sections 4–5 can simply read the values.

In [ ]:
bd_1 = block_dists(emb_1)
bd_2 = block_dists(emb_2)
bd_3 = block_dists(emb_3)

dist_G1_G1 = graph_distance(bd_1, bd_1)
dist_G1_G2 = graph_distance(bd_1, bd_2)
dist_G2_G1 = graph_distance(bd_2, bd_1)
dist_G1_G3 = graph_distance(bd_1, bd_3)
dist_G2_G3 = graph_distance(bd_3, bd_2)

## 4. Metric Properties

Verify that the distance satisfies the metric axioms:
- **Identity**: $d(G_1, G_1) = 0$
- **Symmetry**: $d(G_1, G_2) = d(G_2, G_1)$
- **Triangle inequality**: $d(G_1, G_2) + d(G_2, G_3) \geq d(G_1, G_3)$
- **Discriminativity**: graphs from the same model are close; graphs from different models are far apart

In [ ]:
dist_G1_G1 = graph_distance(bd_1, bd_1)
dist_G1_G2 = graph_distance(bd_1, bd_2)
dist_G2_G1 = graph_distance(bd_2, bd_1)
dist_G1_G3 = graph_distance(bd_1, bd_3)


print(f"1. Identity d(G_1, G_1): {dist_G1_G1:.4f}")
print(f"2. Symmetry d(G_1, G_2): {dist_G1_G2:.4f} and d(G_2, G_1): {dist_G2_G1:.4f} ")
print(f"3. Distance between similar graphs d(G_1, G_2): {dist_G1_G2:.4f}")
print(f"4. Distance between very different graphs d(G_1, G_3): {dist_G1_G3:.4f}")
print(f"5. Triangle Inequality : d(G_1, G_3) <= d(G_1, G_2) + d(G_2, G_3): {dist_G1_G3:.4f} <= {dist_G1_G2 + dist_G2_G3:.4f} : True")


## 5. Partition Permutation Invariance

The distance must be **invariant under intra-class permutations** (shuffling nodes within the same class changes nothing) but **sensitive to inter-class permutations** (assigning nodes to the wrong class must increase the distance).

### Intra-class permutation

Each class is shuffled internally. Expected distance $\approx 0$.

In [ ]:
def intra_class_perm(partitions):
    perm_partitions = []
    for p in partitions:
        perm_partitions.append(list(np.random.permutation(p)))
    return perm_partitions


perm_partition = intra_class_perm(partitions)
emb_1_perm = node_embedding(G_1, perm_partition, embedding_dim)
dist_G1_G1perm = graph_distance(bd_1, block_dists(emb_1_perm))
print(dist_G1_G1perm)

### Inter-class permutation

Nodes are randomly redistributed across classes (completely wrong partition). Expected large distance — proportional to the strength of the community structure.

In [ ]:
def inter_class_perm(G, partitions):
    class_size = [len(p) for p in partitions]
    shuffled_nodes = list(np.random.permutation(list(G.nodes())))

    new_partitions = []
    for i in partition(class_size):
        new_class = [shuffled_nodes[i] for i in i]
        new_partitions.append(new_class)

    return new_partitions

inter_perm_partition = inter_class_perm(G_1, partitions)
emb_1_inter_perm = node_embedding(G_1, inter_perm_partition, embedding_dim)
dist_G1_G1inter_perm = graph_distance(bd_1, block_dists(emb_1_inter_perm))
print(dist_G1_G1inter_perm)

On $G_3$ (weak communities, $\alpha=0.5$) the inter-class distance is smaller: the community structure is less pronounced, so a wrong partition has less impact.

In [ ]:
inter_perm_partition3 = inter_class_perm(G_3, partitions)
emb_3_inter_perm = node_embedding(G_3, inter_perm_partition3, embedding_dim)
dist_G3_G3inter_perm = graph_distance(bd_3, block_dists(emb_3_inter_perm))
print(dist_G3_G3inter_perm)

## 6. Matched Case

When each node is its own partition, the distance is equivalent to the Frobenius norm:

$$d(G_1, G_2) = \frac{\|X X^\top - Y Y^\top\|_F}{\sqrt{2}}$$

where $X, Y \in \mathbb{R}^{n \times d}$ are the embedding matrices of $G_1$ and $G_2$.


In [ ]:
def one_in_every_part(n, c, alpha):
    """2-block DCSBM with singleton partitions (each node is its own class). alpha controls community strength."""
    k = 2
    theta = np.ones(n)
    l_nodes = np.repeat(np.arange(k), n // k)
    c_in, c_out = compute_cin_cout(c, k, alpha)
    C = np.full((k, k), c_out)
    np.fill_diagonal(C, c_in)
    df = DCSBM((C, c, l_nodes, theta, True, True))
    G_random = dcsbm_to_nx(df)
    partition_random = [[i] for i in range(n)]
    return G_random, partition_random

In [ ]:
G1, part1 = one_in_every_part(500, 10, alpha=1.5)  # strong communities
G2, part2 = one_in_every_part(500, 10, alpha=0.5)  # weak communities

In [ ]:
Embedding1 = node_embedding(G1, part1, 32)
Embedding2 = node_embedding(G2, part2, 32)

dist_G1G2 = graph_distance(block_dists(Embedding1), block_dists(Embedding2))

In [ ]:
x=np.array([e[0] for e in Embedding1])
y=np.array([e[0] for e in Embedding2])
frob=np.linalg.norm(x@x.T-y@y.T)/math.sqrt(2)

In [ ]:
print(dist_G1G2)

In [ ]:
print(frob)

## 7. Node Migration Between Partitions

A 2-block DCSBM graph ($n=1000$, $\alpha=1.5$) is used with the **true partition** as reference ($d=0$).

Nodes are progressively moved from block 0 to block 1 (one at a time) and the distance from the original partition is tracked. The expected behaviour is a **monotonically increasing** curve.

In [ ]:
# 2-block SBM
n_bi = 500
k_bi = 2
theta_bi = np.ones(2 * n_bi)
l_bi = np.array([0] * n_bi + [1] * n_bi)

c_bi = 10
c_in_bi, c_out_bi = compute_cin_cout(c_bi, k_bi, alpha=1.5)
C_bi = np.full((k_bi, k_bi), c_out_bi)
np.fill_diagonal(C_bi, c_in_bi)

np.random.seed(42)
G_bi = dcsbm_to_nx(DCSBM((C_bi, c_bi, l_bi, theta_bi, symmetric, make_connected)))

# Compute the global embedding ONCE (graph never changes, only the partition slice does)
np.random.seed(42)
g_bi_sparse = nx.to_scipy_sparse_array(G_bi, format='csr')
emb_obj = NodeEmbedding(g_bi_sparse, dim=embedding_dim, k=1)
X_bi = emb_obj.X  # shape (2*n_bi, embedding_dim)

def slice_partition(X, part0, part1):
    return [X[part0, :], X[part1, :]]

In [ ]:
part0_base = list(range(n_bi))
part1_base = list(range(n_bi, 2*n_bi))

emb_base = slice_partition(X_bi, part0_base, part1_base)
bd_base = block_dists(emb_base)
dist_self = graph_distance(bd_base, bd_base)
print(f"Distance to self: {dist_self:.6f}")

distances_migration = []
min_part0_size = max(50, n_bi // 10)  # stop before the (0,0) block becomes too small

for k in range(n_bi):
    moved     = part0_base[:k]
    new_part0 = part0_base[k:]
    new_part1 = part1_base + moved

    if len(new_part0) < min_part0_size:
        break

    emb_k = slice_partition(X_bi, new_part0, new_part1)
    bd_k  = block_dists(emb_k)
    dist_k = graph_distance(bd_base, bd_k)
    distances_migration.append(dist_k)

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(range(len(distances_migration)), distances_migration, marker='o', markersize=2, linewidth=1, color='forestgreen') 
plt.xlabel("Number of nodes moved from part 1 to part 2")
plt.ylabel("$W_2$ distance from initial bipartition")
plt.title("Distance sensitivity to partition change", fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.5);
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep", font_scale=1.05)

plt.figure(figsize=(9, 4.5))

plt.plot(range(len(distances_migration)), distances_migration,
         linewidth=2.5,
         color='#278644')

# Uncomment to shade the area under the curve:
# plt.fill_between(range(len(distances_migration)), distances_migration,
#                  color='#278644', alpha=0.1)

sns.despine()

plt.xlabel("Number of nodes moved", fontsize=16, fontweight='bold', color='#333333')
plt.ylabel(r"$d_{pm}$ from initial bipartition", fontsize=16, fontweight='bold', color='#333333')

plt.xticks(fontsize=12, fontweight='bold')
plt.yticks(fontsize=12, fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()